# 06 — Modelado tabular y búsqueda de hiperparámetros

**Pivot metodológico**

Esta fase del proyecto NO usa ECG crudo. Trabaja sobre el dataset tabular filtrado generado por `scripts/02_build_filtered_tabular_modeling_dataset.py`, que combina anotaciones (PhysioNet) con `metadata.csv` y deriva features temporales por latido dentro de cada caso (rr_prev, rr_next, hr_inst_from_rr_prev, position_in_case).

El flujo basado en ventaneo de señal ECG (notebooks `04` y `05`, módulos `src/search.py`, scripts `01-03` con sufijo ECG) queda como línea exploratoria histórica.

**Reglas obligatorias (codificadas en `src/tabular_search.py`)**

- Split 80/20 estricto por `case_id` (`make_train_test_group_split_with_coverage`).
- `beat_type`, `case_id`, `rhythm_classes`, `bad_signal_quality*`, identificadores y outcomes posteriores quedan fuera del set de predictores (`config.TABULAR_LEAKAGE_COLUMNS`).
- CV interna por grupo (`StratifiedGroupKFold` con fallback a `GroupKFold`).
- Métrica primaria `f1_macro`. Complementarias: `precision_macro`, `recall_macro`, `accuracy`, `balanced_accuracy`, `f1_weighted`.
- Test congelado: se evalúa una sola vez al final.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.evaluation import (
    class_support_per_split,
    classes_missing_in_train,
    confusion_matrix_with_totals,
    per_class_report,
)
from src.modeling import make_train_test_group_split_with_coverage
from src.tabular_search import (
    PRIMARY_SCORING,
    TABULAR_MODEL_NAMES,
    build_cv_splitter,
    classify_features,
    evaluate_on_test,
    extract_feature_importance,
    load_tabular_modeling_dataset,
    run_search_for_model,
)
from src.utils import get_logger, set_seed

set_seed(config.RANDOM_SEED)
logger = get_logger("nbt")
sns.set_theme(context="notebook", style="whitegrid")

## 2. Carga del dataset filtrado

In [ ]:
df = load_tabular_modeling_dataset()
print("shape:", df.shape)
print("cases:", df[config.CASE_ID_COLUMN].nunique())
print("classes:", sorted(df[config.TARGET_COLUMN].unique().tolist()))
df[config.TARGET_COLUMN].value_counts()

## 3. Clasificación de columnas (numéricas / categóricas / leakage)

In [ ]:
cls = classify_features(df)
print(f"numeric: {len(cls['numeric_features'])}")
print(f"categorical: {len(cls['categorical_features'])}")
print(f"leakage excluido: {cls['leakage_excluded']}")
print(f"alta cardinalidad excluido: {cls['high_cardinality_excluded']}")
print(f"constantes excluidas: {cls['constant_excluded']}")

## 4. Split 80/20 por `case_id` (con diagnóstico de cobertura)

In [ ]:
numeric = cls["numeric_features"]
categorical = cls["categorical_features"]
X_df = df[numeric + categorical]
y = df[config.TARGET_COLUMN].to_numpy()
groups = df[config.CASE_ID_COLUMN].to_numpy()

train_idx, test_idx, split_info = make_train_test_group_split_with_coverage(
    X_df, y, groups, test_size=0.2, random_state=config.RANDOM_SEED, max_attempts=200,
)
X_train, X_test = X_df.iloc[train_idx], X_df.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

assert set(groups_train).isdisjoint(set(groups_test))
print(json.dumps({k: v for k, v in split_info.items() if k not in ("train_groups", "test_groups")}, indent=2, default=str))
print(f"train groups: {len(split_info['train_groups'])} | test groups: {len(split_info['test_groups'])}")
print(f"train rows:   {len(train_idx)} | test rows:   {len(test_idx)}")

In [ ]:
class_support_per_split(y_train, y_test)

## 5. CV builder por grupo

In [ ]:
N_SPLITS = 5
cv, cv_name, n_splits_eff = build_cv_splitter(groups_train=groups_train, y_train=y_train, n_splits=N_SPLITS)
print("splitter:", cv_name, "| n_splits efectivo:", n_splits_eff)

## 6. Búsqueda de hiperparámetros

Para ejecución interactiva en notebook se usan números modestos (`N_ITER=5`). Para la corrida completa, usar `scripts/03_run_tabular_hyperparameter_search.py` con `--n-iter 30 --n-splits 5`.

In [ ]:
MODELS_TO_RUN = list(TABULAR_MODEL_NAMES)
N_ITER = 5
N_JOBS = -1

results = []
fitted_models = {}
for model_name in MODELS_TO_RUN:
    logger.info("===== %s =====", model_name)
    try:
        res = run_search_for_model(
            model_name=model_name,
            X_train=X_train,
            y_train=y_train,
            groups_train=groups_train,
            numeric_features=numeric,
            categorical_features=categorical,
            cv=cv,
            n_iter=N_ITER,
            random_state=config.RANDOM_SEED,
            n_jobs=N_JOBS,
        )
        res = evaluate_on_test(res, X_test, y_test)
        fitted_models[model_name] = res.best_estimator
        results.append({
            "model": res.model,
            "status": res.status,
            "best_cv_score_primary": res.best_cv_score_primary,
            **res.cv_metrics,
            **res.test_metrics,
            "fit_seconds": res.fit_seconds,
            "best_params": res.best_params,
        })
    except Exception as exc:  # noqa: BLE001
        logger.error("%s ERROR: %s", model_name, exc)
        results.append({"model": model_name, "status": "error", "error": str(exc)})

results_df = pd.DataFrame(results)
results_df.round(3)

## 7. Mejor modelo en test

In [ ]:
ok = results_df.loc[results_df["status"] == "ok"].dropna(subset=[f"test_{PRIMARY_SCORING}"]).copy()
if ok.empty:
    print("Sin ganador válido.")
    winner = None
else:
    winner = ok.sort_values(f"test_{PRIMARY_SCORING}", ascending=False).iloc[0]
    print(f"Mejor modelo: {winner['model']}")
    print(f"test_{PRIMARY_SCORING} = {winner[f'test_{PRIMARY_SCORING}']:.3f}")

In [ ]:
if winner is not None:
    best_est = fitted_models[winner["model"]]
    y_pred = best_est.predict(X_test)
    print("Reporte por clase:")
    print(per_class_report(y_test, y_pred).round(3).to_string())
    print()
    print("Matriz de confusión absoluta + totales:")
    print(confusion_matrix_with_totals(y_test, y_pred).to_string())

## 8. Matriz de confusión (visual)

In [ ]:
if winner is not None:
    from sklearn.metrics import confusion_matrix
    labels = sorted(set(pd.Series(y_test).unique()) | set(pd.Series(y_pred).unique()), key=str)
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(1.0 + len(labels), 0.8 + 0.8 * len(labels)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, cbar=False, ax=ax)
    ax.set_title(f"Matriz de confusión absoluta — {winner['model']}")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 9. Importancia / coeficientes de features (mejor modelo)

In [ ]:
if winner is not None:
    fi = extract_feature_importance(fitted_models[winner["model"]])
    if fi is None:
        print("No se pudo extraer importancia/coeficientes para este modelo.")
    else:
        print("Top 20:")
        print(fi.head(20).to_string(index=False))

## 10. Notas

- Este notebook usa **`N_ITER=5`** para ejecución interactiva. Para la corrida completa con `n_iter=30` y persistencia de outputs, ejecutar:
  ```bash
  python scripts/03_run_tabular_hyperparameter_search.py --n-iter 30 --n-splits 5
  ```
- Antes de interpretar el `test_f1_macro` global, mirar la sección 4 (soporte por clase). Si una clase tiene support muy bajo en test, su f1 individual es ruidoso.
- `beat_type` permanece en el parquet como columna descriptiva pero **nunca** entra al modelo (`classify_features` lo descarta por leakage).